# Train π0 (`pi0`) on RunPod — FR5 pick-and-place

π0 is a flow-matching VLA: a PaliGemma-2B vision-language model conditions a 300M-parameter action expert that predicts 50-step action chunks via flow matching.

This notebook runs the repo's shared trainer (`common/train.py --policy pi0`) on a RunPod
GPU pod, end to end: environment → auth → dataset → config → sanity check → training →
monitoring → checkpoint upload.

## Pod setup (before opening this notebook)
| | |
|---|---|
| **Template** | RunPod **PyTorch 2.x** (CUDA ≥ 12.1) |
| **GPU** | 48 GB (A40 / A6000 / L40S) recommended · 24 GB (4090) works with `expert_only` · 80 GB (A100/H100) = full finetune |
| **Disk** | ≥ 60 GB volume (~5 GB PaliGemma + dataset + checkpoints) |

## One-time prerequisites
1. **HF token** (read + write) whose account has **accepted the PaliGemma license**:
   <https://huggingface.co/google/paligemma-3b-pt-224> — the weights are gated; nothing runs without this.
2. **Dataset on the Hub** — push once from wherever the 150-episode dataset lives:
   ```bash
   python tools/push_dataset_hf.py --root lerobot_dataset --repo <you>/fr5-pick-place-lerobot
   ```
3. If the GitHub repo is private: a GitHub token with repo-read scope.

## 1 · Parameters

Everything you might want to change lives here. The three that matter most:

| variable | meaning |
|---|---|
| `MEMORY_MODE` | `auto` picks by VRAM. `full` = finetune everything · `freeze_vision` = freeze SigLIP · `expert_only` = freeze the whole 2B VLM, train only the 300M action expert + projections |
| `BATCH_SIZE` | `None` = auto by VRAM (see cheat-sheet below) |
| `PROPRIO_MODE` | `full` / `dropout` / `none` — the proprioception benchmark axis |

**VRAM cheat-sheet** (bf16 + gradient checkpointing are always on):

| GPU | memory mode | batch |
|---|---|---|
| 24 GB | `expert_only` | 2 |
| 48 GB | `freeze_vision` | 4 |
| 80 GB | `full` | 4–8 |

> `expert_only` is not just a memory fallback — with 150 episodes it's also the
> least-overfitting choice, mirroring how Octo is finetuned head-only in this repo.

In [ ]:
import os

HF_TOKEN        = os.environ.get("HF_TOKEN", "")        # hf_... (read+write, PaliGemma licence accepted)
HF_DATASET_REPO = "<you>/fr5-pick-place-lerobot"        # pushed with tools/push_dataset_hf.py
GIT_URL         = "https://github.com/SreevaatsavB/fairino-fr5-policies.git"
GIT_TOKEN       = os.environ.get("GIT_TOKEN", "")       # only if the repo is private
GIT_BRANCH      = "main"

POLICY       = "pi0"
MEMORY_MODE  = "auto"      # auto | full | freeze_vision | expert_only
BATCH_SIZE   = None        # None -> picked from GPU VRAM
MAX_EPOCHS   = 100
PROPRIO_MODE = "full"      # full | dropout | none

WORKSPACE = "/workspace"
REPO_DIR  = f"{WORKSPACE}/fairino-fr5-policies"
DATA_DIR  = f"{WORKSPACE}/lerobot_dataset"

assert HF_TOKEN.startswith("hf_"), "set HF_TOKEN (env var or paste above) — PaliGemma is gated"
print("parameters set")

## 2 · Clone the repo

Idempotent — re-running pulls the latest `main` instead of re-cloning.

In [ ]:
import subprocess, pathlib

if not pathlib.Path(REPO_DIR, ".git").exists():
    url = GIT_URL.replace("https://", f"https://{GIT_TOKEN}@") if GIT_TOKEN else GIT_URL
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

print(subprocess.check_output(["git", "-C", REPO_DIR, "log", "--oneline", "-3"], text=True))

## 3 · Install dependencies

`lerobot==0.5.1` pins the torch/transformers stack — first install takes ~3–5 min,
re-runs are fast. (pip may replace the pod template's preinstalled torch; that's expected.)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", f"{REPO_DIR}/requirements.txt"], check=True)

import torch, lerobot, transformers
print(f"torch {torch.__version__} · lerobot {lerobot.__version__} · transformers {transformers.__version__}")

## 4 · HuggingFace auth + gated-weights check

Fails fast with the license URL if the token can't access PaliGemma —
much better than dying 20 minutes later mid-download.

In [ ]:
from huggingface_hub import login, whoami, auth_check
from huggingface_hub.errors import GatedRepoError

login(token=HF_TOKEN, add_to_git_credential=False)
print("logged in as:", whoami()["name"])

try:
    auth_check("google/paligemma-3b-pt-224")
    print("PaliGemma licence OK — gated weights accessible")
except GatedRepoError:
    raise SystemExit("PaliGemma is gated for this token — accept the licence at "
                     "https://huggingface.co/google/paligemma-3b-pt-224 and re-run")

## 5 · GPU check → auto memory mode + batch size

Reads the pod's VRAM and resolves `MEMORY_MODE`/`BATCH_SIZE` if they were left on auto.

In [ ]:
import torch

assert torch.cuda.is_available(), "no CUDA GPU — pick a GPU pod"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
assert torch.cuda.is_bf16_supported(), "bf16 unsupported — use an Ampere or newer GPU"
print(f"{name}  {vram:.0f} GB")

if MEMORY_MODE == "auto":
    MEMORY_MODE = "expert_only" if vram < 40 else ("freeze_vision" if vram < 70 else "full")
if BATCH_SIZE is None:
    BATCH_SIZE = {"expert_only": 4 if vram >= 40 else 2,
                  "freeze_vision": 4, "full": 4 if vram >= 70 else 2}[MEMORY_MODE]
print(f"memory_mode = {MEMORY_MODE}   batch_size = {BATCH_SIZE}")

## 6 · Pull the dataset from the Hub

In [ ]:
from huggingface_hub import snapshot_download
import json, pathlib

snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=DATA_DIR)
info = json.loads(pathlib.Path(DATA_DIR, "meta", "info.json").read_text())
print(f"episodes = {info['total_episodes']}   frames = {info['total_frames']}   "
      f"fps = {info['fps']}   robot = {info.get('robot_type')}")

### 6b · Eyeball the data

A wrist-cam frame and the 7-D action traces of one episode — if these look wrong
(black frames, flat traces), stop here and fix the dataset before spending GPU-hours.

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

df = pd.read_parquet(pathlib.Path(DATA_DIR, "data/chunk-000/file-000.parquet"))
ep = df[df.episode_index == 0]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
frames = sorted(pathlib.Path(DATA_DIR, "frames").rglob("ep-000/*.jpg"))
if frames:
    ax[0].imshow(mpimg.imread(frames[len(frames) // 2])); ax[0].axis("off")
    ax[0].set_title(f"wrist cam, mid-episode ({len(frames)} frames)")
acts = pd.DataFrame(ep["action"].tolist())
acts.plot(ax=ax[1], legend=False, title=f"episode 0 action traces (7-D, {len(ep)} steps)")
plt.tight_layout(); plt.show()
print("task:", ep["task"].iloc[0] if "task" in ep else "(see meta/tasks.parquet)")

## 7 · Write the pod-local training config

Starts from `policies/pi0/config.yaml` and patches only what's pod-specific:
dataset path, batch size, epochs, the memory mode, and a separate checkpoint dir.

In [ ]:
import yaml, pathlib

base = yaml.safe_load(pathlib.Path(REPO_DIR, "policies", POLICY, "config.yaml").read_text())

base["dataset"]["root"] = DATA_DIR
base["model"].update({
    "dtype": "bfloat16",
    "gradient_checkpointing": True,
    "freeze_vision_encoder": MEMORY_MODE == "freeze_vision",
    "train_expert_only":     MEMORY_MODE == "expert_only",
    "proprio_mode": PROPRIO_MODE,
})
base["training"].update({
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "checkpoint_dir": f"policies/{POLICY}/checkpoints_runpod",
    "device": "cuda",
})

cfg_path = pathlib.Path(REPO_DIR, "policies", POLICY, "config.runpod.yaml")
cfg_path.write_text(yaml.safe_dump(base, sort_keys=False))
print("wrote", cfg_path, "\n")
print(yaml.safe_dump({"model-memory": {k: base["model"][k] for k in
      ("dtype", "gradient_checkpointing", "freeze_vision_encoder", "train_expert_only")},
      "training": base["training"]}, sort_keys=False))

## 8 · Sanity check — build the model + time one step *(optional but recommended)*

Downloads PaliGemma (~5 GB, first run only), builds π0 with your memory mode,
prints trainable vs frozen parameter counts, and times one forward+backward on a real batch.
If this OOMs, fix it **now** (cell 1 → `expert_only` / smaller batch) instead of mid-run.
~2–4 min on first run; skip freely on re-runs.

In [ ]:
import sys, time, yaml, pathlib, importlib.util, torch

sys.path.insert(0, f"{REPO_DIR}/common")
from dataset import FR5Dataset

cfg = yaml.safe_load(pathlib.Path(REPO_DIR, "policies", POLICY, "config.runpod.yaml").read_text())
spec = importlib.util.spec_from_file_location("policy_model", f"{REPO_DIR}/policies/{POLICY}/model.py")
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

ds = FR5Dataset(DATA_DIR, chunk_size=cfg["dataset"]["chunk_size"], use_image=True)
model = mod.build_model(cfg, ds.get_stats(), torch.device("cuda"))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"trainable {trainable/1e6:.0f}M   frozen {frozen/1e6:.0f}M   ({MEMORY_MODE})")

b = torch.utils.data.default_collate([ds[i] for i in range(BATCH_SIZE)])
t0 = time.time()
loss, _, _ = model(b["observation.state"].cuda(), b["action"].cuda(),
                   b["action_is_pad"].cuda(), b["observation.images.wrist_cam"].cuda(),
                   task=["pick up the block and place it in the bin"] * BATCH_SIZE)
loss.backward()
torch.cuda.synchronize()
step_s = time.time() - t0
steps_per_epoch = max(1, len(ds) // BATCH_SIZE)
print(f"1 step = {step_s:.2f}s  →  ~{step_s * steps_per_epoch / 60:.1f} min/epoch, "
      f"~{step_s * steps_per_epoch * MAX_EPOCHS / 3600:.1f} h for {MAX_EPOCHS} epochs")
print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB / {vram:.0f} GB")

del model, b, loss
torch.cuda.empty_cache()

## 9 · Launch training

Runs under `nohup` so a dropped Jupyter/browser connection does **not** kill it.
First run downloads PaliGemma before step 1 (if cell 8 was skipped) — watch the log.

In [ ]:
import subprocess, sys, pathlib

log = pathlib.Path(REPO_DIR, "train_pi0.log")
cmd = (f"cd {REPO_DIR} && nohup {sys.executable} common/train.py "
       f"--policy {POLICY} --config policies/{POLICY}/config.runpod.yaml "
       f"> {log} 2>&1 & echo $!")
pid = subprocess.check_output(cmd, shell=True, text=True).strip()
print(f"training started  pid={pid}")
print(f"log: {log}")
print(f"stop it with:  !kill {pid}")

## 10 · Monitor — re-run these two cells any time

In [ ]:
# live log tail
import pathlib
log = pathlib.Path(REPO_DIR, "train_pi0.log")
print("".join(log.read_text().splitlines(keepends=True)[-20:]))

In [ ]:
# loss curves from metrics.csv (written at the end of every epoch)
import pandas as pd, pathlib
import matplotlib.pyplot as plt

csv = pathlib.Path(REPO_DIR, "policies", POLICY, "checkpoints_runpod", "metrics.csv")
if csv.exists():
    m = pd.read_csv(csv)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
    m.plot(x="epoch", y=["train_l1", "val_l1"], ax=ax[0], title="flow-matching loss")
    m.plot(x="epoch", y="train_val_gap", ax=ax[1], title="train/val gap (overfitting watch)")
    m.plot(x="epoch", y="grad_norm", ax=ax[2], title="grad norm")
    plt.tight_layout(); plt.show()
    best = m.loc[m.val_l1.idxmin()]
    print(f"best val_l1 = {best.val_l1:.4f} @ epoch {int(best.epoch)}  "
          f"({len(m)} epochs logged)")
else:
    print("metrics.csv not written yet — appears after epoch 1")

## 11 · Ship checkpoints to the Hub (when training is done)

Uploads `best.pt` + `metrics.csv` to a private model repo, so the checkpoint survives
pod termination. Deploy on the robot box with `python common/deploy.py --checkpoint best.pt`.

In [ ]:
from huggingface_hub import HfApi, whoami
import pathlib

ckpt_dir = pathlib.Path(REPO_DIR, "policies", POLICY, "checkpoints_runpod")
repo = f"{whoami()['name']}/fr5-{POLICY}-{MEMORY_MODE}"

api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=str(ckpt_dir), repo_id=repo,
                  allow_patterns=["best.pt", "metrics.csv"],
                  commit_message=f"{POLICY} {MEMORY_MODE} bs={BATCH_SIZE} epochs={MAX_EPOCHS}")
print(f"uploaded -> https://huggingface.co/{repo}")

## Troubleshooting

**CUDA OOM** — in order of preference:
1. `MEMORY_MODE = "expert_only"` (freezes the 2B VLM; the 300M expert still learns the task)
2. halve `BATCH_SIZE` (set it explicitly, e.g. `BATCH_SIZE = 1`)
3. both

Both variables live in **cell 1 (Parameters)** — edit them there, kill the old run
(`!kill <pid>`), then re-run cells **1 → 5 → 7 → 9** so the new values flow into the
auto-selection, the generated config, and the relaunch (cell 8 to re-verify memory).

**`GatedRepoError` / 403 on PaliGemma** — the HF account hasn't accepted the license,
or the token lacks read scope.

**Pod restarted mid-run** — checkpoints land every `save_every` (10) epochs in
`policies/pi0/checkpoints_runpod/`. The trainer has no resume flag yet, so a relaunch
starts from scratch weights — for spot pods, consider a lower `MAX_EPOCHS` per run.

**Throughput sanity** — cell 8 prints the measured step time and a full-run estimate.
Watch the first ~50 steps of the log for a *decreasing* `train_l1` before walking away.